# Brandy Roberts Art — LoRA Trainer

Trains a custom Stable Diffusion XL LoRA on your uploaded artwork so the app can generate new pieces in your signature style.

**How to use this notebook:**
1. Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU → Save**
2. Click **Runtime → Run all**
3. When Cell 3 pauses, paste the dataset URL from your Brandy Roberts Art app (the one starting with `https://...supabase.co/storage/...` — NOT this Colab page's URL)
4. Wait 30–60 minutes (Cell 5 is the long one)
5. Copy the final URL printed at the bottom and paste it back into the app's Train tab

Free Colab T4 GPU is sufficient. If you hit GPU limits, try again in a few hours.

Trigger word for your prompts after training: **brandystyle**

In [ ]:
# Cell 1 — Verify GPU is attached
!nvidia-smi

In [ ]:
# Cell 2 — Install dependencies (this takes ~3 minutes; ignore the red sentence-transformers warning, it's harmless)
!pip install -q diffusers==0.27.0 transformers==4.40.0 accelerate==0.29.0 peft==0.10.0 bitsandbytes
!pip install -q xformers==0.0.25 datasets safetensors
!git clone https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!pip install -q -r requirements.txt

In [ ]:
# Cell 3 — Get dataset URL from Brandy Roberts Art
# IMPORTANT: Paste the DATASET URL from the app's Train tab, NOT this Colab page's URL.
# It should start with 'https://' and end with '.zip' (or contain 'supabase.co/storage')
import os, requests, zipfile, io

DATASET_URL = input('Paste your training dataset URL from Brandy Roberts Art: ').strip()
SESSION_ID = input('Paste your session ID (optional, press Enter to skip): ').strip() or 'session'

# Validate the URL looks right
if 'colab.research.google.com' in DATASET_URL:
    raise ValueError('That looks like the Colab page URL — you need the DATASET URL from the app instead. Check the Train tab in Brandy Roberts Art for the dataset link.')

print(f'Downloading dataset from {DATASET_URL[:80]}...')
r = requests.get(DATASET_URL)
if r.status_code != 200:
    raise ValueError(f'Download failed with status {r.status_code}. Check that the URL is correct and not expired.')

# Extract the zip
os.makedirs('/content/dataset/10_artwork', exist_ok=True)
try:
    zipfile.ZipFile(io.BytesIO(r.content)).extractall('/content/dataset/10_artwork')
except zipfile.BadZipFile:
    raise ValueError('The downloaded file is not a valid zip. Make sure you copied the DATASET URL (the zip file), not the Colab page URL.')

image_count = len([f for f in os.listdir('/content/dataset/10_artwork') if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))])
print(f'Successfully extracted {image_count} images')

In [ ]:
# Cell 4 — Auto-caption images using BLIP
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

proc = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
mdl = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base').to('cuda')

for f in os.listdir('/content/dataset/10_artwork'):
    if not f.lower().endswith(('.jpg','.jpeg','.png','.webp')):
        continue
    img = Image.open(f'/content/dataset/10_artwork/{f}').convert('RGB')
    inputs = proc(img, return_tensors='pt').to('cuda')
    cap = proc.decode(mdl.generate(**inputs)[0], skip_special_tokens=True)
    caption_path = f'/content/dataset/10_artwork/{os.path.splitext(f)[0]}.txt'
    with open(caption_path, 'w') as out:
        out.write(f'brandystyle, {cap}')
    print(f'{f}: brandystyle, {cap}')

print('\nCaptions generated. Trigger word for prompts: brandystyle')

In [ ]:
# Cell 5 — Train the SDXL LoRA (this is the long step, ~30-60 min)
!accelerate launch --num_cpu_threads_per_process=2 sdxl_train_network.py \
  --pretrained_model_name_or_path='stabilityai/stable-diffusion-xl-base-1.0' \
  --train_data_dir='/content/dataset' \
  --output_dir='/content/output' \
  --output_name='brandyroberts_lora' \
  --resolution='1024,1024' \
  --network_module=networks.lora \
  --network_dim=32 --network_alpha=16 \
  --learning_rate=1e-4 --train_batch_size=1 \
  --max_train_epochs=10 --mixed_precision='fp16' \
  --save_every_n_epochs=2 --save_model_as=safetensors \
  --optimizer_type='AdamW8bit' --xformers \
  --gradient_checkpointing --cache_latents

In [ ]:
# Cell 6 — Upload your trained LoRA to a public host so Brandy Roberts Art can use it
import requests

with open('/content/output/brandyroberts_lora.safetensors', 'rb') as f:
    r = requests.post('https://file.io', files={'file': f})

lora_url = r.json().get('link')
print('\n' + '=' * 60)
print('YOUR BRANDY ROBERTS ART LORA IS READY')
print('=' * 60)
print(f'\nCopy this URL and paste it into Brandy Roberts Art:\n')
print(lora_url)
print(f'\nTrigger word for your prompts: brandystyle')
print(f'Example prompt: "brandystyle, abstract terracotta botanical, soft warm light"')
print('=' * 60)